<div align="center">

# Week 06: Estimation and Filtering: The Kalman Filter, State Estimation and Bad Data

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-06) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/Week06_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `lab_rng`, the random generator of the interactive lab, and records the evening of the battery from 17:00 to 21:00. The power readings carry an offset of -5 kW, and the voltage readings carry a noise of 0.01. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

def lab_rng(seed):
    """The random generator of the interactive lab, so that the Colab notebook draws exactly the same readings."""
    s = seed & 0xFFFFFFFF
    def uniform():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (s | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    def normal():
        u = v = 0.0
        while u == 0.0:
            u = uniform()
        while v == 0.0:
            v = uniform()
        return math.sqrt(-2 * math.log(u)) * math.cos(2 * math.pi * v)
    return uniform, normal

# The battery of the campus (Week 2) on the evening of Week 1, from 17:00 to 21:00, one value a minute
E_KWH, ETA, DT_H = 500.0, 0.95, 1 / 60
EVENING_KW = np.repeat([52.9, 130.5, 135.7, 39.9], 60)        # the true discharge power in kW
OFFSET_KW, SIGMA_P, SIGMA_Z = -5.0, 2.0, 0.01                  # the current sensor reads 5 kW too little
SOC_TRUE = np.zeros(241)
SOC_TRUE[0] = 0.90
for k in range(240):
    SOC_TRUE[k + 1] = SOC_TRUE[k] - EVENING_KW[k] * DT_H / (ETA * E_KWH)
_, normal = lab_rng(6)
P_MEAS, Z_SOC = np.zeros(240), np.zeros(240)
for k in range(240):                                           # the recorded evening, the same as on the bench
    P_MEAS[k] = EVENING_KW[k] + OFFSET_KW + SIGMA_P * normal()
    Z_SOC[k] = SOC_TRUE[k + 1] + SIGMA_Z * normal()
print("Setup complete. Running in Colab:", IN_COLAB, "| state of charge", SOC_TRUE[0], "at 17:00 and",
      round(SOC_TRUE[-1], 4), "at 21:00 | readings:", len(P_MEAS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 6.1": (lambda: 7.0, 1e-9),
    "Exercise 6.2": (lambda: -1 / 11, 1e-9),
    "Exercise 6.3": (lambda: 12.0, 1e-9),
    "Exercise A": (lambda: (49.98 / 0.01 ** 2 + 50.01 / 0.02 ** 2) / (1 / 0.01 ** 2 + 1 / 0.02 ** 2), 1e-6),
    "Exercise B": (lambda: rms(kalman_1d(3e-6)[0]), 1e-9),
    "Exercise C": (lambda: float(kalman_2d()[1][179]), 1e-9),
    "Exercise D": (lambda: wls(fault_readings('car park ratio'))['J'], 1e-6),
    "Exercise E": (lambda: float(STAGE6['smallest offset of the B3 feeder meter that is detected (kW)']), 1e-9),
}

## Python step 6: Matrices and the Kalman filter

The Kalman filter and the state estimation of this week work with vectors and matrices. This step builds them with NumPy, multiplies them with the operator @, transposes and inverts them and runs a short filter loop &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "Z_SOC" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Vectors and matrices

In [ ]:
# Python practice: a small example for learning the language
x = np.array([0.9, -5.0])                      # a state: the state of charge and an offset in kW
A = np.array([[1.0, 0.0000351], [0.0, 1.0]])    # a matrix of two rows and two columns
print(x.shape, A.shape)
print(A)

`np.array` turns a list into a vector and a list of lists into a matrix. The attribute `.shape` gives the size: Two values for the vector, two rows and two columns for the matrix.

### Matrix products with @

In [ ]:
# Python practice: a small example for learning the language
print(A @ x)                                    # the matrix times the vector
H = np.array([[1.0, 0.0]])
print(H @ x)                                    # a row that picks the first element

The operator `@` multiplies as in linear algebra: Each element of the result is a row of `A` times the vector. The ordinary `*` would multiply element by element, which is a different operation.

### Transpose and inverse

In [ ]:
# Python practice: a small example for learning the language
P = np.array([[4.0, 1.0], [1.0, 3.0]])
print(P.T)                                      # the transpose: rows become columns
P_inv = np.linalg.inv(P)
print(np.round(P_inv @ P, 10))                  # the inverse times the matrix gives the identity

`.T` swaps rows and columns, and `np.linalg.inv` computes the inverse. A matrix times its inverse is the identity matrix, with ones on its diagonal. For a system of linear equations, `np.linalg.solve(A, b)` is faster and more accurate than the inverse.

### A filter loop

In [ ]:
# Python practice: a small example for learning the language
readings = [21.4, 20.8, 21.6, 21.1, 21.3]
estimate, n = readings[0], 1
for z in readings[1:]:
    n += 1
    estimate += (z - estimate) / n             # the running mean, corrected by each new reading
    print(f"{z:5.1f} -> {estimate:.3f}")

Each new reading corrects the estimate by a fraction of the difference, here 1/n. This running mean is the simplest recursive filter, because it never stores the old readings. The Kalman filter works the same way, with a fraction that it computes from the uncertainties.

**Quick check.** Why does the correction become smaller with each reading?

<details><summary>Answer</summary>

The fraction 1/n shrinks: The more readings the mean already holds, the less one new reading can change it.

</details>

### Exercises for Python step 6

**Exercise 6.1.** Multiply the matrix `[[2, 0], [1, 3]]` with the vector `[1, 2]` and store the second element of the result in `v_61`.

In [ ]:
v_61 = None   # your answer

check("Exercise 6.1", v_61)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
v_61 = float((np.array([[2, 0], [1, 3]]) @ np.array([1, 2]))[1])

check("Exercise 6.1", v_61)

**Exercise 6.2.** Store the element in the first row and second column of the inverse of `P` in `p_62`.

In [ ]:
p_62 = None   # your answer

check("Exercise 6.2", p_62)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_62 = float(np.linalg.inv(P)[0, 1])

check("Exercise 6.2", p_62)

**Exercise 6.3.** Run the running mean over the readings `[10, 12, 14]` and store the final estimate in `m_63`.

In [ ]:
m_63 = None   # your answer

check("Exercise 6.3", m_63)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_63, n = 10.0, 1
for z in [12, 14]:
    n += 1
    m_63 += (z - m_63) / n

check("Exercise 6.3", m_63)

## 1. Combining measurements &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#ref-1)&#93;

*Lecture: [Estimation: Combining noisy measurements](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#estimation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "P_inv" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Two sensors that measure the same quantity disagree because of their noise. Python combines them with weights that depend on their uncertainty and checks with 100,000 simulated readings that the combination is better than either sensor alone.

**From the formulas to the code.** The weights $w_i = 1/\sigma_i^2$, the weighted mean $\hat{x} = \sum w_i x_i / \sum w_i$ and its standard deviation $1/\sqrt{\sum w_i}$ are computed in `combine`. For two sensors, the same mean is $x_A + K (x_B - x_A)$ with $K = \sigma_A^2/(\sigma_A^2 + \sigma_B^2)$, the gain `k`.

In [ ]:
def combine(values, sigmas):
    """The inverse-variance weighted mean of several measurements and its standard deviation."""
    w = 1 / np.asarray(sigmas, float) ** 2
    return float(np.sum(w * np.asarray(values, float)) / np.sum(w)), float(1 / np.sqrt(np.sum(w)))

x_hat, s_hat = combine([21.4, 20.8], [0.3, 0.5])
k = 0.3 ** 2 / (0.3 ** 2 + 0.5 ** 2)
print(f"two temperature sensors: {x_hat:.3f} C with a standard deviation of {s_hat:.3f} K")
print(f"the same as 21.4 + K * (20.8 - 21.4) with K = {k:.4f}: {21.4 + k * (20.8 - 21.4):.3f} C")

rng = np.random.default_rng(RANDOM_STATE)
sens_a = 21.0 + rng.normal(0, 0.3, 100000)     # 100,000 readings of a true 21.0 C by sensor A
sens_b = 21.0 + rng.normal(0, 0.5, 100000)     # and by sensor B
w_a, w_b = 1 / 0.3 ** 2, 1 / 0.5 ** 2
comb = (w_a * sens_a + w_b * sens_b) / (w_a + w_b)
print(f"spread: sensor A {sens_a.std():.3f} K, sensor B {sens_b.std():.3f} K, weighted mean {comb.std():.3f} K, "
      f"plain mean {((sens_a + sens_b) / 2).std():.3f} K")

**Python note.** `combine` accepts lists of any length, so the same function combines three or ten sensors. The simulation draws both sensors 100,000 times and shows that the weighted mean spreads less than either sensor and less than the plain mean.

**Exercise A.** Two frequency meters read 49.98 Hz with a standard deviation of 10 mHz and 50.01 Hz with 20 mHz. Store their weighted mean in Hz in `f_a`.

In [ ]:
f_a = None   # your answer

check("Exercise A", f_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
f_a, _ = combine([49.98, 50.01], [0.01, 0.02])

check("Exercise A", f_a)

In [ ]:
# Figure: the spread of two sensors and of their weighted mean
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.linspace(19.5, 22.5, 121)
ax.hist(sens_b, bins=bins, color="#C9D1D8", label="sensor B, 0.5 K")
ax.hist(sens_a, bins=bins, color="#C98510", alpha=0.6, label="sensor A, 0.3 K")
ax.hist(comb, bins=bins, color="#1E6FA8", alpha=0.7, label=f"weighted mean, {comb.std():.3f} K")
ax.set_xlabel("reading (C)")
ax.set_ylabel("count")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why is the weighted mean better than the plain mean of the two sensors? What happens to the weights if sensor B is replaced by a much worse one?

## 2. The Kalman filter: Your runs on the battery bench &#91;[4](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#ref-4)&#93;

*Lecture: [The Kalman filter: Predict and correct](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#kalman)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "combine" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The battery management system knows its state of charge only through a biased current sensor and a noisy voltage. Python compares coulomb counting, the voltage alone and a Kalman filter on the recorded evening of 17:00 to 21:00, and checks your runs on the bench.

**From the formulas to the code.** The prediction $\hat{x}^- = \hat{x} - P_{meas}\,\Delta t/(\eta E)$ and $P^- = P + Q$, the gain $K = P^-/(P^- + R)$ and the correction $\hat{x} = \hat{x}^- + K(z - \hat{x}^-)$ with $P = (1 - K)P^-$ are the four lines of the loop in `kalman_1d`. `coulomb` keeps only the prediction.

In [ ]:
def rms(est):
    """The root-mean-square error of an estimate of the evening against the true state of charge."""
    return float(np.sqrt(np.mean((np.asarray(est) - SOC_TRUE[1:]) ** 2)))

def coulomb(x0=0.95):
    """Coulomb counting: the estimate follows the measured power alone."""
    x, out = x0, []
    for p in P_MEAS:
        x -= p * DT_H / (ETA * E_KWH)
        out.append(x)
    return np.array(out)

def kalman_1d(Q, R=SIGMA_Z ** 2, x0=0.95, P0=0.05 ** 2):
    """A Kalman filter for the state of charge: predict with the measured power, correct with the voltage reading."""
    x, P, est, gain = x0, P0, [], []
    for p, z in zip(P_MEAS, Z_SOC):
        x -= p * DT_H / (ETA * E_KWH)          # predict
        P += Q
        K = P / (P + R)                        # the Kalman gain
        x += K * (z - x)                       # correct
        P *= 1 - K
        est.append(x)
        gain.append(K)
    return np.array(est), np.array(gain)

print(f"coulomb counting: RMS error {rms(coulomb()):.4f}, error at 21:00 {coulomb()[-1] - SOC_TRUE[-1]:+.4f}")
print(f"voltage readings: RMS error {rms(Z_SOC):.4f}, error at 21:00 {Z_SOC[-1] - SOC_TRUE[-1]:+.4f}")
for Q in [1e-8, 1e-7, 1e-6, 1e-5]:
    est, gain = kalman_1d(Q)
    print(f"Kalman, Q = {Q:.0e}: RMS error {rms(est):.4f}, error at 21:00 {est[-1] - SOC_TRUE[-1]:+.4f}, "
          f"final gain {gain[-1]:.4f}")

**Python note.** `x -= ...` is short for `x = x - ...`, and `P *= 1 - K` for `P = P * (1 - K)`. The filter keeps only two numbers between the minutes, the estimate and its variance, and never stores the old readings.

**Exercise B.** Store the root-mean-square (RMS) error of the Kalman filter with `Q=3e-6` in `rms_b`.

In [ ]:
rms_b = None   # your answer

check("Exercise B", rms_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rms_b = rms(kalman_1d(3e-6)[0])

check("Exercise B", rms_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lab.html) replays the same evening on the battery management bench. Paste the copied rows into `MY_RUNS`, and the code computes each of your runs again from the same readings.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs on the bench"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4, 5], "method": ["coulomb", "voltage", "kalman", "kalman", "kalman"],
                         "q": [None, None, 1e-8, 1e-6, 1e-5], "rms_error": [0.0722, 0.0107, 0.0102, 0.0037, 0.0050]})
    source = "a simulated student"
def model_rms(r):
    if r.method == "coulomb":
        return rms(coulomb())
    if r.method == "voltage":
        return rms(Z_SOC)
    if r.method == "kalman":
        return rms(kalman_1d(r.q)[0])
    return rms(kalman_2d()[0])
runs["model_rms"] = [round(model_rms(r), 4) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "method", "q", "rms_error", "model_rms"]].to_string(index=False))

In [ ]:
# Figure: the state of charge on the evening, true, counted, read and filtered
t = 17 + np.arange(1, 241) / 60
est, gain = kalman_1d(1e-6)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(t, Z_SOC, ".", ms=2, color="#C9D1D8", label="voltage readings")
ax[0].plot(t, coulomb(), color="#B53F2E", label="coulomb counting")
ax[0].plot(t, est, color="#1E6FA8", label="Kalman, Q = 1e-6")
ax[0].plot(t, SOC_TRUE[1:], color="#1B2430", lw=1, ls="--", label="true")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("state of charge")
ax[0].legend(fontsize=8)
ax[1].plot(t, gain, color="#1E6FA8")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("Kalman gain K")
ax[1].set_yscale("log")
plt.show()

**Observe and reflect.** Which value of Q gave you the smallest error on the bench, and why do a smaller and a larger Q both do worse? Why does the gain start near 1 and then fall?

## 3. The Kalman filter with matrices: The offset of the sensor &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#ref-1)&#93;

*Lecture: [The Kalman filter with matrices](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#kalman-matrix)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "kalman_1d" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The offset of the current sensor is a constant that coulomb counting cannot see. Python adds it to the state as a second number and lets a Kalman filter with matrices estimate both, the state of charge and the offset.

**From the formulas to the code.** The prediction $\hat{\mathbf{x}}^- = A\hat{\mathbf{x}} + B u$ and $P^- = A P A^T + Q$, the gain $K = P^- H^T (H P^- H^T + R)^{-1}$ and the correction with $P = (I - K H) P^-$ are the lines of the loop in `kalman_2d`. The state is the state of charge and the offset in kW, so `A`, `P` and `Q` are matrices of 2 by 2, and `H` picks the state of charge.

In [ ]:
def kalman_2d(q_soc=1e-8, q_off=1e-6, R=SIGMA_Z ** 2, x0=0.95, P0=0.05 ** 2, P_off0=100.0):
    """A Kalman filter for the state of charge and the offset of the current sensor."""
    c = DT_H / (ETA * E_KWH)
    A = np.array([[1.0, c], [0.0, 1.0]])               # the offset makes the true discharge larger than measured
    H = np.array([[1.0, 0.0]])                         # the voltage measures the state of charge only
    Qm = np.diag([q_soc, q_off])
    x, P = np.array([x0, 0.0]), np.diag([P0, P_off0])
    est, off, sd_off = [], [], []
    for p, z in zip(P_MEAS, Z_SOC):
        x = A @ x + np.array([-p * c, 0.0])            # predict
        P = A @ P @ A.T + Qm
        S = H @ P @ H.T + R                            # the variance of the innovation, 1 by 1
        K = P @ H.T @ np.linalg.inv(S)                 # the gain, 2 by 1
        x = x + K @ (np.array([z]) - H @ x)            # correct
        P = (np.eye(2) - K @ H) @ P
        est.append(x[0])
        off.append(x[1])
        sd_off.append(math.sqrt(P[1, 1]))
    return np.array(est), np.array(off), np.array(sd_off)

est2, off, sd_off = kalman_2d()
print(f"Kalman with the offset: RMS error {rms(est2):.4f}, error at 21:00 {est2[-1] - SOC_TRUE[-1]:+.4f}")
for hour, k in [(18, 59), (19, 119), (20, 179), (21, 239)]:
    print(f"  offset estimated at {hour}:00: {off[k]:6.2f} kW +- {sd_off[k]:.2f} kW (true {OFFSET_KW} kW)")

**Python note.** `A @ P @ A.T` multiplies three matrices from left to right. `np.linalg.inv(S)` inverts a matrix of 1 by 1 here, and with several measurements the same line inverts a larger matrix.

**Exercise C.** Store the offset in kW that `kalman_2d` estimates at 20:00, the element 179 of `off`, in `b_c`.

In [ ]:
b_c = None   # your answer

check("Exercise C", b_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
b_c = float(off[179])

check("Exercise C", b_c)

In [ ]:
# Figure: the estimated offset of the current sensor with its uncertainty
t = 17 + np.arange(1, 241) / 60
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.fill_between(t, off - 2 * sd_off, off + 2 * sd_off, color="#DCEBF5", label="two standard deviations")
ax.plot(t, off, color="#1E6FA8", label="estimated offset")
ax.axhline(OFFSET_KW, color="#B53F2E", ls="--", label="true offset")
ax.set_ylim(-20, 10)
ax.set_xlabel("hour")
ax.set_ylabel("offset of the current sensor (kW)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why can the filter find the offset at all, although no sensor measures it? What would happen if the voltage readings stopped for an hour?

## 4. State estimation and bad data: Your runs in the control room &#91;[6](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#ref-6)&#93;

*Lecture: [Bad data: Residuals and the chi-square test](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#bad-data)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "kalman_2d" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Thirteen meters report six feeder powers at 19:00, each with its own uncertainty. Python estimates the feeders by weighted least squares, tests the residuals and finds a faulty meter, as the control room of the campus does. One such fault is an open current transformer (CT).

**From the formulas to the code.** The estimate $\hat{\mathbf{x}} = (H^T W H)^{-1} H^T W \mathbf{z}$ with $W = R^{-1}$, the residuals $\mathbf{r} = \mathbf{z} - H\hat{\mathbf{x}}$, the test value $J = \mathbf{r}^T W \mathbf{r}$ and the normalized residuals $|r_i|/\sqrt{\Omega_{ii}}$ with $\Omega = R - H (H^T W H)^{-1} H^T$ are computed in `wls`. The threshold is the 99 percent quantile of the chi-square distribution, `chi2.ppf`.

In [ ]:
NAMES = ["B1 feeder", "B2 feeder", "B3 feeder", "B4 feeder", "B5 feeder", "Car park feeder",
         "B1 building", "B2 building", "B3 building", "B4 building", "B5 building", "Car park chargers", "Main meter"]
X_TRUE = np.array([500 * 180 / 700, 500 * 120 / 700, 500 * 150 / 700, 500 * 90 / 700, 500 * 160 / 700, 88.0])
H_SE = np.vstack([np.eye(6), np.eye(6), np.ones((1, 6))])     # feeder meters, building meters, main meter
SIGMA_SE = np.array([1.5] * 6 + [2.5] * 6 + [3.0])
Z_SE = np.array([130.2, 88.4, 103.3, 64.1, 115.8, 90.0, 130.2, 89.5, 107.9, 65.7, 114.7, 85.3, 585.5])

def wls(z, keep=None):
    """Weighted least squares with the chi-square test and the normalized residuals; keep marks the meters used."""
    keep = np.ones(len(z), bool) if keep is None else keep
    H, s, zk = H_SE[keep], SIGMA_SE[keep], np.asarray(z, float)[keep]
    W = np.diag(1 / s ** 2)
    G = H.T @ W @ H
    x = np.linalg.solve(G, H.T @ W @ zk)
    r = zk - H @ x
    omega = s ** 2 - np.einsum("ij,jk,ik->i", H, np.linalg.inv(G), H)
    rn = np.abs(r) / np.sqrt(omega)
    J = float(r @ W @ r)
    dof = int(keep.sum()) - H.shape[1]
    return dict(x=x, J=J, threshold=float(chi2.ppf(0.99, dof)), rn=rn, keep=keep)

def fault_readings(fault, offset=0.0):
    """The readings with one fault: open CT of B3, wrong ratio of the car park, frozen library, main offset."""
    z = Z_SE.copy()
    if fault == "B3 open CT":
        z[2] = 0.0
    elif fault == "car park ratio":
        z[5] = round(z[5] * 1.4, 1)
    elif fault == "library frozen":
        z[7] = 45.0
    elif fault == "main offset":
        z[12] -= 30.0
    elif fault == "B3 offset":
        z[2] += offset
    return z

def identify(z):
    """Removes the meter with the largest normalized residual while the test fails."""
    res = wls(z)
    if res["J"] <= res["threshold"]:
        return res, None, res
    i = int(np.flatnonzero(res["keep"])[np.argmax(res["rn"])])
    keep = res["keep"].copy()
    keep[i] = False
    return res, NAMES[i], wls(z, keep)

for fault in ["none", "B3 open CT", "car park ratio", "library frozen", "main offset"]:
    first, bad, after = identify(fault_readings(fault))
    print(f"{fault:15}: J = {first['J']:8.2f} against {first['threshold']:.2f} -> "
          + (f"bad meter {bad}, J after removal {after['J']:.2f}, B3 estimated at {after['x'][2]:.2f} kW" if bad
             else f"no bad data, B3 estimated at {first['x'][2]:.2f} kW"))
for e in range(6, 14):
    res = wls(fault_readings("B3 offset", e))
    print(f"B3 feeder meter {e:+3d} kW: J = {res['J']:6.2f} -> {'detected' if res['J'] > res['threshold'] else 'not detected'}")

**Python note.** `H_SE[keep]` keeps only the rows of the meters in use, so removing a meter is a change of one `True` to `False`. `np.einsum` computes only the diagonal of $H (H^T W H)^{-1} H^T$, which is all that the normalized residuals need.

**Exercise D.** Store the test value J when the car park feeder meter reads 40 percent too much in `j_d`.

In [ ]:
j_d = None   # your answer

check("Exercise D", j_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
j_d = wls(fault_readings('car park ratio'))['J']

check("Exercise D", j_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lab.html) opens the control room of the campus. Paste the copied rows into `MY_METERS`, and the code repeats each of your runs with the same readings.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_METERS = """
"""
if MY_METERS.strip():
    meters = pd.read_csv(io.StringIO(MY_METERS.strip()))
    source = "your own runs in the control room"
else:
    meters = pd.DataFrame({"run": [1, 2, 3, 4], "fault": ["none", "B3 open CT", "B3 offset", "B3 offset"],
                           "offset_kw": [0, 0, 10, 11], "J": [8.04, 1626.32, 17.05, 19.55]})
    source = "a simulated student"
meters["model_J"] = [round(wls(fault_readings(r.fault, r.offset_kw))["J"], 2) for r in meters.itertuples()]
print("Data:", source)
print(meters[["run", "fault", "offset_kw", "J", "model_J"]].to_string(index=False))

In [ ]:
# Figure: the normalized residuals without a fault and with the open CT of B3
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=False)
for axis, fault in zip(ax, ["none", "B3 open CT"]):
    res = wls(fault_readings(fault))
    axis.bar(range(13), res["rn"], color=["#B53F2E" if v > 3 else "#1E6FA8" for v in res["rn"]])
    axis.axhline(3, color="#56616D", ls=":")
    axis.set_xticks(range(13))
    axis.set_xticklabels([n.replace(" feeder", " f").replace(" building", " b") for n in NAMES], rotation=60, fontsize=7)
    axis.set_ylabel("normalized residual")
    axis.set_title(f"{fault}: J = {res['J']:.1f}", fontsize=9)
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the open CT of B3 also produce large residuals at other meters? Why can the building meters and the main meter still locate it?

## 5. The campus microgrid, stage 6

*Lecture: [The campus microgrid, stage 6: Estimating the state of the microgrid](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#the-campus-microgrid-stage-6)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "identify" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 6 gives the controllers of the campus estimates instead of raw readings. The battery management system tracks its state of charge and the offset of its current sensor with a Kalman filter. The control room estimates the six feeders from thirteen meters and rejects a faulty meter.

In [ ]:
STAGE6 = {}
STAGE6["coulomb counting: error at 21:00"] = round(float(coulomb()[-1] - SOC_TRUE[-1]), 4)
STAGE6["voltage readings: RMS error"] = round(rms(Z_SOC), 4)
q_values = [1e-8, 1e-7, 1e-6, 3e-6, 1e-5]
errors = [rms(kalman_1d(q)[0]) for q in q_values]
STAGE6["Kalman filter: best Q"] = q_values[int(np.argmin(errors))]
STAGE6["Kalman filter: smallest RMS error"] = round(min(errors), 4)
est2, off, sd_off = kalman_2d()
STAGE6["Kalman with the offset: RMS error"] = round(rms(est2), 4)
STAGE6["Kalman with the offset: offset at 21:00 (kW)"] = round(float(off[-1]), 2)
STAGE6["state estimation: J without a fault"] = round(wls(Z_SE)["J"], 2)
first, bad, after = identify(fault_readings("B3 open CT"))
STAGE6["open CT of B3: J"] = round(first["J"], 2)
STAGE6["open CT of B3: meter found"] = bad
STAGE6["open CT of B3: B3 estimated after removal (kW)"] = round(float(after["x"][2]), 2)
STAGE6["smallest offset of the B3 feeder meter that is detected (kW)"] = next(
    e for e in range(1, 40) if wls(fault_readings("B3 offset", e))["J"] > wls(Z_SE)["threshold"])
for key, value in STAGE6.items():
    print(f"{key:<62} {value}")

**Python note.** `next(...)` returns the first value of a generator, here the first whole offset in kW whose test value exceeds the threshold.

In [ ]:
# Figure: the state of charge with three estimators and the residual test against the offset of the B3 meter
t = 17 + np.arange(1, 241) / 60
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(t, coulomb() - SOC_TRUE[1:], color="#B53F2E", label="coulomb counting")
ax[0].plot(t, kalman_1d(1e-6)[0] - SOC_TRUE[1:], color="#C98510", label="Kalman, Q = 1e-6")
ax[0].plot(t, est2 - SOC_TRUE[1:], color="#1E6FA8", label="Kalman with the offset")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("error of the state of charge")
ax[0].legend(fontsize=8)
offsets = np.arange(0, 21)
ax[1].plot(offsets, [wls(fault_readings("B3 offset", e))["J"] for e in offsets], "o-", color="#1E6FA8", ms=4)
ax[1].axhline(wls(Z_SE)["threshold"], color="#B53F2E", ls="--", label="threshold, 99 percent")
ax[1].set_xlabel("offset of the B3 feeder meter (kW)")
ax[1].set_ylabel("test value J")
ax[1].legend(fontsize=8)
plt.show()

**Exercise E.** Store the smallest offset of the B3 feeder meter that the test detects, as kept in `STAGE6`, in `e_e`.

In [ ]:
e_e = None   # your answer

check("Exercise E", e_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e_e = STAGE6["smallest offset of the B3 feeder meter that is detected (kW)"]

check("Exercise E", e_e)

**Observe and reflect.** Coulomb counting would believe 0.236 at 21:00 instead of 0.144. What would the battery do with this error during the outage of Week 3?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE6" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-06/"
               "NB06_estimation_filtering.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever a quantity changes slowly and is measured with noise, a Kalman filter can track it. The functions of this week filter a simulated signal after one change of a name.

The signals and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "gps_position"
# choose: gps_position, room_temperature, heart_rate, water_level, tyre_pressure, satellite_clock
APPS = {  # (quantity, unit, change per step q_std, noise of a reading r_std, steps)
    "gps_position": ("position of a walker from GPS", "m", 0.5, 4.0, 600),
    "room_temperature": ("temperature of a room", "C", 0.02, 0.3, 720),
    "heart_rate": ("heart rate from a wrist sensor", "1/min", 0.8, 6.0, 600),
    "water_level": ("level of a tank from an ultrasonic sensor", "cm", 0.1, 2.0, 500),
    "tyre_pressure": ("pressure of a tyre", "kPa", 0.05, 3.0, 400),
    "satellite_clock": ("offset of a clock", "ns", 0.3, 5.0, 800)}
desc, unit, q_std, r_std, n = APPS[APP]
rng = np.random.default_rng(RANDOM_STATE)
truth = np.cumsum(rng.normal(0, q_std, n))      # a random walk
z = truth + rng.normal(0, r_std, n)              # its noisy readings
x, P, est = z[0], r_std ** 2, []
for zk in z:
    P += q_std ** 2                              # predict: the quantity may have changed
    K = P / (P + r_std ** 2)
    x += K * (zk - x)                            # correct with the reading
    P *= 1 - K
    est.append(x)
est = np.array(est)
print(f"{APP}: {desc}")
print(f"  RMS error of the raw readings {np.sqrt(np.mean((z - truth) ** 2)):.3g} {unit}, "
      f"of the Kalman estimate {np.sqrt(np.mean((est - truth) ** 2)):.3g} {unit}")
print(f"  final gain {K:.3f}: each reading moves the estimate {100 * K:.1f} percent of the way")

**Python note.** The model is a random walk: The quantity changes a little at each step, with the standard deviation `q_std`. The ratio of `q_std` to `r_std` decides the gain: A quantity that barely changes is averaged over many readings.

**Observe and reflect.** For which signal does the filter help most, and why? What would happen if `q_std` were set ten times too small?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a class `KalmanFilter` with the methods `predict` and `update` for any matrices A, H, Q and R, and test it against `kalman_2d`. |
| Embedded systems | Run `kalman_1d` in single precision with `np.float32` and compare the result. Where would a small microcontroller lose accuracy? |
| Internet of Things | The voltage readings arrive over a radio that loses 30 percent of them. Skip the correction for each lost reading and measure the effect on the error. |
| Computer networks | The building meters reach the control room with a delay of up to 5 s. How would a state estimator handle readings from slightly different times? |
| Control engineering | Feed the estimate of `kalman_2d` instead of the counted state of charge into the rule of Week 3 that keeps the reserve for an outage. What changes? |
| Data science | Estimate Q and R from the readings alone by maximising the likelihood of the innovations over a grid of values. |
| Cybersecurity | An attacker adds the same 20 kW to the B3 feeder meter, to the B3 building meter and to the main meter. Show that J does not change, and explain why. |
| Artificial intelligence | Train a small regression that predicts the state of charge from the last ten voltage readings and compare it with the Kalman filter. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-06).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>